In [3]:

import pandas as pd
import numpy as np
import itertools
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import xgboost as xgb
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# load data


In [6]:
X_train = pd.read_csv('property_data_train_x_engineered.csv').drop(columns=['ListPrice', 'OriginalListPrice', 'ListingKey'])
X_test = pd.read_csv('property_data_test_x_engineered.csv').drop(columns=['ListPrice', 'OriginalListPrice', 'ListingKey'])

y_train = pd.read_csv('property_data_train_y_engineered.csv')
y_test = pd.read_csv('property_data_test_y_engineered.csv')

In [7]:
# Create the XGBoost model
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

# Perform light hyperparameter tuning
params = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 0.01, 0.001],
    'n_estimators': [100, 200, 300]
}

from sklearn.model_selection import GridSearchCV
xgb_grid = GridSearchCV(xgb_model, params, cv=5, scoring='r2', n_jobs=-1)
xgb_grid.fit(X_train, y_train)

# Get the best hyperparameters and model
best_xgb_params = xgb_grid.best_params_
best_xgb_model = xgb_grid.best_estimator_

In [8]:
# Create the LightGBM model
lgbm_model = LGBMRegressor(random_state=42)

# Perform light hyperparameter tuning
params = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 0.01, 0.001],
    'n_estimators': [100, 200, 300]
}

from sklearn.model_selection import GridSearchCV
lgbm_grid = GridSearchCV(lgbm_model, params, cv=5, scoring='r2', n_jobs=-1)
lgbm_grid.fit(X_train, y_train)

# Get the best hyperparameters and model
best_lgbm_params = lgbm_grid.best_params_
best_lgbm_model = lgbm_grid.best_estimator_

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.114244 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2465
[LightGBM] [Info] Number of data points in the train set: 118391, number of used features: 118
[LightGBM] [Info] Start training from score 1198903.027709
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

In [9]:
# Evaluate the best XGBoost model
xgb_y_pred = best_xgb_model.predict(X_test)
xgb_mse = mean_squared_error(y_test, xgb_y_pred)
xgb_r2 = r2_score(y_test, xgb_y_pred)
print(f'XGBoost: MSE={xgb_mse:.2f}, R^2={xgb_r2:.2f}')

# Evaluate the best LightGBM model
lgbm_y_pred = best_lgbm_model.predict(X_test)
lgbm_mse = mean_squared_error(y_test, lgbm_y_pred)
lgbm_r2 = r2_score(y_test, lgbm_y_pred)
print(f'LightGBM: MSE={lgbm_mse:.2f}, R^2={lgbm_r2:.2f}')

XGBoost: MSE=98089861120.00, R^2=0.89
LightGBM: MSE=100753322209.33, R^2=0.88
